# Multi-Armed Bandit

**Last updated:** 2026-04-09 12:00 PT

**Track:** Learning | **Sub-Ability:** Reinforcement learning

Can the model learn optimal strategies from reward feedback?
Pre/post learning framework: observe reward histories, identify best actions.

**Difficulty grid:** num arms × reward clarity × 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import sqlite3
import json
import time
import urllib.request
import urllib.error

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_short_answer(response: str) -> str:
    """Extract a short answer (last short line, stripped of formatting)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines: return ''
    for line in reversed(lines):
        clean = line.strip('"\'\u2018\u2019\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\u2018\u2019\u201c\u201d').rstrip('.!?').lower().strip()

def extract_category(response: str) -> str:
    """Extract a category letter (A-D) from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        m = re.search(r'\b([A-D])\b', line)
        if m and len(line) < 30: return m.group(1)
    for line in reversed(lines):
        m = re.search(r'([A-D])', line)
        if m: return m.group(1)
    return ''

In [ ]:
# === Multi-Armed Bandit: synthetic reward history generation ===
# Each arm has a hidden reward probability. We generate a history of plays and outcomes.
# The model must identify the best arm from the history.

ARM_LABELS = ['A', 'B', 'C', 'D']

def generate_bandit_history(rng, num_arms, clarity, num_plays):
    """Generate a bandit scenario with controlled reward probabilities and a play history.

    Args:
        rng: random.Random instance
        num_arms: number of arms (2, 3, or 4)
        clarity: 'clear', 'moderate', or 'ambiguous'
        num_plays: number of plays in the history

    Returns:
        arms: list of arm labels
        probs: dict of arm -> reward probability
        history: list of (arm, reward) tuples
        best_arm: the arm with highest empirical reward rate
    """
    arms = ARM_LABELS[:num_arms]

    # Generate reward probabilities based on clarity
    if clarity == 'clear':
        # One arm is clearly dominant (e.g. 0.8 vs 0.2-0.3)
        best_idx = rng.randint(0, num_arms - 1)
        probs = {}
        for i, arm in enumerate(arms):
            if i == best_idx:
                probs[arm] = rng.uniform(0.75, 0.90)
            else:
                probs[arm] = rng.uniform(0.10, 0.30)
    elif clarity == 'moderate':
        # Moderate differences (e.g. 0.65 vs 0.35-0.45)
        best_idx = rng.randint(0, num_arms - 1)
        probs = {}
        for i, arm in enumerate(arms):
            if i == best_idx:
                probs[arm] = rng.uniform(0.60, 0.75)
            else:
                probs[arm] = rng.uniform(0.25, 0.45)
    else:  # ambiguous
        # Very close probabilities — requires careful statistical reasoning
        base = rng.uniform(0.40, 0.55)
        best_idx = rng.randint(0, num_arms - 1)
        probs = {}
        for i, arm in enumerate(arms):
            if i == best_idx:
                probs[arm] = base + rng.uniform(0.05, 0.10)
            else:
                probs[arm] = base + rng.uniform(-0.05, 0.02)

    # Generate play history — roughly balanced across arms with some variation
    history = []
    for _ in range(num_plays):
        arm = rng.choice(arms)
        reward = 1 if rng.random() < probs[arm] else 0
        history.append((arm, reward))

    # Compute empirical reward rates
    arm_rewards = {a: [] for a in arms}
    for arm, reward in history:
        arm_rewards[arm].append(reward)

    empirical_rates = {}
    for arm in arms:
        if arm_rewards[arm]:
            empirical_rates[arm] = sum(arm_rewards[arm]) / len(arm_rewards[arm])
        else:
            empirical_rates[arm] = 0.0

    best_arm = max(arms, key=lambda a: empirical_rates[a])

    return arms, probs, history, empirical_rates, best_arm


def format_history(arms, history):
    """Format play history as a readable string."""
    lines = []
    for i, (arm, reward) in enumerate(history, 1):
        outcome = 'WIN (reward 1)' if reward == 1 else 'LOSS (reward 0)'
        lines.append(f'  Round {i}: Chose {arm} -> {outcome}')
    return '\n'.join(lines)


def format_summary_stats(arms, history):
    """Compute and format summary statistics for the history."""
    arm_rewards = {a: [] for a in arms}
    for arm, reward in history:
        arm_rewards[arm].append(reward)

    lines = []
    for arm in arms:
        plays = len(arm_rewards[arm])
        wins = sum(arm_rewards[arm])
        rate = wins / plays if plays > 0 else 0.0
        lines.append(f'  Arm {arm}: {plays} plays, {wins} wins, win rate = {rate:.0%}')
    return '\n'.join(lines)


# Difficulty axes
NUM_ARMS = [2, 3, 4]
CLARITIES = ['clear', 'moderate', 'ambiguous']
# History length scales with num_arms to give enough data
HISTORY_LENGTHS = {2: 20, 3: 30, 4: 40}
SEEDS = 3

In [ ]:
# === Dataset ===
rows = []; tid = 0

for num_arms in NUM_ARMS:
    for clarity in CLARITIES:
        for seed in range(SEEDS):
            rng = random.Random(seed * 100 + num_arms * 10)
            num_plays = HISTORY_LENGTHS[num_arms]
            arms, probs, history, empirical_rates, best_arm = generate_bandit_history(
                rng, num_arms, clarity, num_plays)

            history_text = format_history(arms, history)
            arm_list = ', '.join(arms)
            difficulty_label = f'{num_arms}arms_{clarity}'

            material = (
                f'You are playing a {num_arms}-armed bandit game with arms: {arm_list}.\n'
                f'Each arm gives a reward of 1 (win) or 0 (loss) when pulled.\n'
                f'Here is the history of {num_plays} plays:\n\n'
                f'{history_text}\n'
            )

            # Store true probabilities and empirical rates for analysis
            prob_str = ', '.join(f'{a}={probs[a]:.2f}' for a in arms)
            emp_str = ', '.join(f'{a}={empirical_rates[a]:.2f}' for a in arms)

            rows.append({
                'task_id': tid, 'seed': seed, 'num_arms': num_arms,
                'clarity': clarity, 'difficulty_label': difficulty_label,
                'material': material, 'test_input': material,
                'expected': best_arm, 'item_desc': f'{num_arms}arms_{clarity}_s{seed}',
                'true_probs': prob_str, 'empirical_rates': emp_str,
                'num_plays': num_plays, 'arms': arm_list,
            })
            tid += 1

dataset = pd.DataFrame(rows)
print(f'Dataset: {len(dataset)} items')
print(dataset[['task_id', 'difficulty_label', 'expected', 'item_desc', 'true_probs', 'empirical_rates']].to_string(index=False))

In [ ]:
DB_PATH = 'bandit_runs.db'
db = sqlite3.connect(DB_PATH)
db.execute('DROP TABLE IF EXISTS runs')
db.execute('''
    CREATE TABLE runs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT, model TEXT, task_id TEXT,
        num_arms INTEGER, clarity TEXT, difficulty_label TEXT,
        seed INTEGER, item_desc TEXT,
        test_input TEXT, expected TEXT,
        true_probs TEXT, empirical_rates TEXT, num_plays INTEGER, arms TEXT,
        pre_raw TEXT, pre_extracted TEXT, pre_correct INTEGER,
        study_notes TEXT,
        post_raw TEXT, post_extracted TEXT, post_correct INTEGER,
        pre_time_s REAL, study_time_s REAL, post_time_s REAL
    )
''')
db.commit()
print(f'SQLite ready (fresh): {DB_PATH}')

In [ ]:
PRE_P = """You are playing a multi-armed bandit game. Your goal is to maximize reward.

{material}

Based on this history, which arm should you choose next to maximize reward?

Reply with ONLY the letter (e.g. A, B, C, or D). Nothing else."""

STUDY_P = """You are playing a multi-armed bandit game. Your goal is to maximize reward.

{material}

Analyze this data carefully:
1. Calculate the average reward (win rate) for each arm.
2. Identify which arm has the highest win rate.
3. Note any patterns or changes over time in the reward history.
4. Determine the best strategy going forward.

Write a clear analysis with your calculations."""

POST_P = """You previously analyzed a multi-armed bandit game and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Here is the reward history again:
{material}

Based on your analysis and this history, which arm should you choose next to maximize reward?

Reply with ONLY the letter (e.g. A, B, C, or D). Nothing else."""

## Evaluation

In [ ]:
@kbench.task(name='multi_armed_bandit',
             description='Pre/post learning multi-armed bandit — identify best arm from reward history')
def multi_armed_bandit(llm, material: str, test_input: str, expected: str,
                       num_arms: int, clarity: str, difficulty_label: str,
                       seed: int, item_desc: str, true_probs: str,
                       empirical_rates: str, num_plays: int, arms: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    tid = f'{num_arms}arms_{clarity}_{seed}'

    # --- PRE: answer without structured analysis ---
    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(material=material))
    pre_time = time.time() - t0
    pre_extracted = extract_category(pre_raw)
    pre_correct = pre_extracted == expected

    # --- STUDY: structured analysis of reward data ---
    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(material=material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    # --- POST: answer with notes from study phase ---
    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, material=material))
    post_time = time.time() - t0
    post_extracted = extract_category(post_raw)
    post_correct = post_extracted == expected

    try:
        db.execute(
            """INSERT INTO runs (timestamp,model,task_id,num_arms,clarity,difficulty_label,
               seed,item_desc,test_input,expected,true_probs,empirical_rates,num_plays,arms,
               pre_raw,pre_extracted,pre_correct,
               study_notes,post_raw,post_extracted,post_correct,
               pre_time_s,study_time_s,post_time_s)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            (ts,model_name,tid,int(num_arms),clarity,difficulty_label,int(seed),item_desc,
             test_input,expected,true_probs,empirical_rates,int(num_plays),arms,
             pre_raw,pre_extracted,int(pre_correct),
             notes,post_raw,post_extracted,int(post_correct),
             pre_time,study_time,post_time))
        db.commit()
    except Exception as e: print(f'  [SQLite warn] {e}')

    b,l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
    print(f'[{difficulty_label:16s}] expected="{expected}"  pre="{pre_extracted}"({b})  post="{post_extracted}"({l})')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

runs = multi_armed_bandit.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                    n_jobs=1, timeout=3600, max_attempts=1)
print(f'\nDone: {len(runs.as_dataframe())} items')

## Results

In [ ]:
results = pd.read_sql('SELECT * FROM runs ORDER BY task_id', db)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc
room = 1.0 - pre_acc
eff = gain / room if room > 0 else 0.0

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print(f'Learning efficiency:    {eff:.1%}')
print()

# Per difficulty label
print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:20s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per clarity
print()
print('By Clarity:')
print('-' * 60)
for clarity in CLARITIES:
    sub = results[results['clarity'] == clarity]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {clarity:20s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per num_arms
print()
print('By Num Arms:')
print('-' * 60)
for na in NUM_ARMS:
    sub = results[results['num_arms'] == na]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {na} arms              pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per item
print()
print('Per-item detail:')
print('-' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    delta = '  HELPED' if not row['pre_correct'] and row['post_correct'] else \
            '  HURT' if row['pre_correct'] and not row['post_correct'] else ''
    print(f'  [{row["difficulty_label"]:16s}] expected="{row["expected"]}"  '
          f'pre="{row["pre_extracted"]}"({b})  post="{row["post_extracted"]}"({l}){delta}')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'True probs: {row["true_probs"]}')
    print(f'Empirical rates: {row["empirical_rates"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Multi-Armed Bandit: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('multi_armed_bandit_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Post-run: upload to BigQuery + Google Sheets via REST API ===
# Safe to do after benchmark — SDK can break, doesn't matter.

results_upload = pd.read_sql('SELECT * FROM runs', db)
db.close()
print(f'SQLite closed. {len(results_upload)} rows to upload.\n')

auth_token = None
gcp_project = None
try:
    import google.auth
    import google.auth.transport.requests
    creds, project = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    auth_token = creds.token
    gcp_project = project
    print(f'Authenticated. Project: {gcp_project}')
except Exception as e:
    print(f'Google auth not available: {e}')

BQ_DATASET = 'benchmark_runs'

# --- BigQuery ---
if auth_token and gcp_project:
    BQ_TABLE = DB_PATH.replace('_runs.db', '')
    try:
        import urllib.parse
        ds_url = f'https://api.kaggle.com/api/v1/bigquery/projects/{gcp_project}/datasets'
        ds_body = json.dumps({'datasetReference': {'datasetId': BQ_DATASET}, 'location': 'US'}).encode()
        req = urllib.request.Request(ds_url, data=ds_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        schema_fields = [{'name': c, 'type': 'STRING' if results_upload[c].dtype == 'object' else
                          'INTEGER' if 'correct' in c or c in ('seed','id','num_arms','num_plays') else 'FLOAT'}
                         for c in results_upload.columns]
        create_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables'
        create_body = json.dumps({'tableReference': {'projectId': gcp_project, 'datasetId': BQ_DATASET, 'tableId': BQ_TABLE},
                                  'schema': {'fields': schema_fields}}).encode()
        req = urllib.request.Request(create_url, data=create_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        table_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables/{BQ_TABLE}/insertAll'
        rows_json = [{'json': {c: str(v) if pd.notna(v) else '' for c, v in row.items()}} for _, row in results_upload.iterrows()]
        insert_body = json.dumps({'rows': rows_json}).encode()
        req = urllib.request.Request(table_url, data=insert_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'BigQuery: {len(results_upload)} rows -> {gcp_project}.{BQ_DATASET}.{BQ_TABLE}')
    except Exception as e:
        print(f'BigQuery failed (non-fatal): {e}')

# --- Google Sheets ---
if auth_token:
    SHEET_NAME = f'Learning Bench — {BQ_TABLE} Runs'
    try:
        import urllib.parse
        search_url = ('https://www.googleapis.com/drive/v3/files'
                      f'?q=name%3D%27{urllib.parse.quote(SHEET_NAME)}%27+and+mimeType%3D%27application/vnd.google-apps.spreadsheet%27'
                      '&fields=files(id,webViewLink)')
        req = urllib.request.Request(search_url, headers={'Authorization': f'Bearer {auth_token}'})
        resp = json.loads(urllib.request.urlopen(req).read())
        files = resp.get('files', [])
        if files:
            sid = files[0]['id']
        else:
            create_url = 'https://sheets.googleapis.com/v4/spreadsheets'
            req = urllib.request.Request(create_url, data=json.dumps({'properties': {'title': SHEET_NAME}}).encode(),
                                         method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            resp = json.loads(urllib.request.urlopen(req).read())
            sid = resp['spreadsheetId']
            header = list(results_upload.columns)
            body = json.dumps({'values': [header]}).encode()
            req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                         data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            urllib.request.urlopen(req)

        data_rows = [[str(v) if pd.notna(v) else '' for v in row] for _, row in results_upload.iterrows()]
        body = json.dumps({'values': data_rows}).encode()
        req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                     data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'Sheets: {len(results_upload)} rows -> "{SHEET_NAME}"')
    except Exception as e:
        print(f'Sheets failed (non-fatal): {e}')

print(f'\nDone. SQLite: {DB_PATH}')